# Figure 4 — Q3: how far does confounding carry a correction before it breaks?

Batch-phenotype confounding is a designed factor here, swept from 0 to 1. Every panel asks the same question at a different angle: where does each method stop working?

Each panel has its own cell and renders on its own, sized to suit it alone. The last cell assembles all of them at the submission size (180 x 230 mm) and writes the PDF.

The panel functions live in `figures/fig4.R` and are **sourced, not copied** — edit them there and re-run the cell, so the notebook and `snakemake` produce the same figure.

## Setup

In [ ]:
# Panels come from the script, so the notebook and the submitted figure cannot drift apart.
# main() is guarded, so sourcing runs nothing.
if (basename(getwd()) == "figures") setwd("..")
stopifnot("run this from benchmark/ or benchmark/figures/" = file.exists("config/methods.tsv"))
source("figures/fig4.R")

results_dir <- "results"

## Data

In [ ]:
scored <- read_scores(file.path(results_dir, "scores.tsv"))
simulated <- scored[scored$regime == "sim" & scored$status == "ok", ]
by_scenario <- scored_by_scenario(simulated)
real_by_dataset <- bind_rows(
  family_score(scored[scored$regime == "real" & scored$status == "ok", ], batch_metrics(),
               by = c("dataset", "method")),
  family_score(scored[scored$regime == "real" & scored$status == "ok", ], bio_metrics(),
               by = c("dataset", "method")))
confounding <- dataset_confounding(results_dir, unique(scored$dataset[scored$regime == "real"]))
names(confounding) <- sub("^real/", "", unique(scored$dataset[scored$regime == "real"]))

# Eighteen curves in one palette are eighteen shades nobody can tell apart: the two baselines and
# the four methods that actually move get colour, the rest stay grey.
sweep_scores <- by_scenario[by_scenario$family_scenario == "sweep", ]
highlight <- gradient_highlight(sweep_scores)
colours <- gradient_colours(highlight)

cat("highlighted:", paste(highlight, collapse = ", "), "\n")

## a. Performance along the gradient

Scaled score against designed confounding, split by assay and by metric family. Grey lines are the other twelve methods.

In [ ]:
show_panel(panel_gradient(sweep_scores, highlight, colours), 180, 88)

## b. Overcorrection index

Biological signal lost per unit of batch signal removed. The axis is symmetric because the ratio goes negative, and that is the finding: apparent phenotype signal *grew* while the batch term shrank.

In [ ]:
show_panel(panel_overcorrection(simulated[simulated$family == "sweep", ], highlight, colours), 98, 72)

## c. Breaking point

Where each method first keeps less than half the signal the oracle still shows. Most never do.

In [ ]:
show_panel(panel_breakpoint(simulated[simulated$family == "sweep", ]), 98, 72)

## d. Invented structure in the null scenarios

With no biological signal planted at all, how much phenotype variance does each method manufacture?

In [ ]:
show_panel(panel_invented_signal(simulated[simulated$family == "null" & simulated$method != "oracle", ]), 170, 88)

## e. The same measures on real data

Balanced against confounded real designs, split at Cramer's V = 0.2.

In [ ]:
show_panel(panel_real_anchors(real_by_dataset, confounding), 170, 88)

## Assembled figure

Everything above, in the layout that goes to the journal: 180 x 230 mm, one page.
The panels shrink here — that is the real submission size, and the font sizes above were
chosen for it.

In [ ]:
figure <- (panel_gradient(sweep_scores, highlight, colours) /
           (panel_overcorrection(simulated[simulated$family == "sweep", ], highlight, colours) |
            panel_breakpoint(simulated[simulated$family == "sweep", ])) /
           panel_invented_signal(simulated[simulated$family == "null" & simulated$method != "oracle", ]) /
           panel_real_anchors(real_by_dataset, confounding)) +
  plot_layout(heights = c(1.4, 1, 1.2, 1.1)) +
  plot_annotation(tag_levels = "a") &
  theme(plot.tag = element_text(size = 9, face = "bold"))

save_figure(figure, "results/figures/fig4.pdf", 180, 230)
show_panel(figure, 180, 230, scale = 1.5)